# BarterBench — Kaggle Community Benchmark

**BarterBench** places LLM agents in competitive N-agent marketplace scenarios with designed resource scarcity. Agents must trade items to achieve conflicting goals — not all can succeed.

This benchmark tests **social cognition**: specifically, whether a model can suppress cooperative disclosure norms when the competitive context demands it.

In [ ]:
# Download BarterBench source files directly (no git required)
import urllib.request, os, time

BASE = 'https://raw.githubusercontent.com/JamesEBall/BarterBench/main/'
FILES = ['agent.py', 'engine.py', 'scoring.py', 'bradley_terry.py',
         'elo.py', 'model_registry.py', 'scenario_gen.py', 'solvability.py', 'taxonomy.py']
SCENARIOS = ['gold_rush', 'water_crisis', 'spice_wars', 'grand_bazaar']

os.makedirs('scenarios', exist_ok=True)
bust = int(time.time())  # cache-bust each run
for f in FILES:
    urllib.request.urlretrieve(f'{BASE}{f}?cb={bust}', f)
for s in SCENARIOS:
    urllib.request.urlretrieve(f'{BASE}scenarios/{s}.json?cb={bust}', f'scenarios/{s}.json')

# Install Python dependencies
!pip install -q openai anthropic python-dotenv

In [ ]:
import kaggle_benchmarks as kbench
import json, random
from pathlib import Path

from agent import (
    MarketAgent, HeuristicAgent,
    _build_marketplace_context, _parse_json_response,
    JSON_SCHEMA_INSTRUCTION,
)
from engine import MarketEngine
from scoring import compute_metrics

SCENARIOS_DIR = Path('scenarios')
SCENARIOS = ['gold_rush', 'water_crisis', 'spice_wars']

def load_scenario(name):
    return json.loads((SCENARIOS_DIR / f'{name}.json').read_text())

In [ ]:
class KbenchAgent(MarketAgent):
    """MarketAgent wrapper that routes LLM calls through kbench.llm."""

    def __init__(self, kbench_llm, agent_idx, **kwargs):
        super().__init__(model_name='kbench', agent_idx=agent_idx,
                         backend='cli', **kwargs)
        self._kbench_llm = kbench_llm

    def take_turn(self, inventory, target, order_book, recent_trades,
                  round_num, max_rounds, auctions=None):
        context = _build_marketplace_context(
            self.agent_idx, inventory, target, order_book, recent_trades,
            round_num, max_rounds, strategy_prompt=self.strategy_prompt,
        )
        prompt = f"{context}\n\n{JSON_SCHEMA_INSTRUCTION}\n\nIt's your turn. Choose your action."
        try:
            raw = self._kbench_llm.prompt(prompt).strip()
            if '```' in raw:
                raw = raw.split('```')[1].lstrip('json').strip()
            action = _parse_json_response(raw)
        except Exception:
            action = self._fallback_pass()
        self._record_round_history(action, round_num)
        return action


def run_match(kbench_llm, scenario_name: str, seed: int) -> dict:
    """1 KbenchAgent (agent 0) vs N-1 HeuristicAgents.
    Returns gc (goal completion 0-1) and iss (information security score 0-1).
    """
    scenario = load_scenario(scenario_name)
    num_agents = len(scenario['agents'])

    agents = [KbenchAgent(kbench_llm, agent_idx=0)]
    for i in range(1, num_agents):
        agents.append(HeuristicAgent(agent_idx=i))

    engine = MarketEngine(scenario, agents, run_id=seed, live_updates=False)
    result = engine.run()
    result['scenario_data'] = scenario
    metrics = compute_metrics(result)

    gc  = metrics.get('model_goal_completion', {}).get('kbench', 0.0)
    iss = metrics.get('information_security_score', {}).get('kbench', 0.0)
    return {'gc': gc, 'iss': iss, 'scenario': scenario_name}

In [ ]:
# Single task = single discrete score on the leaderboard
# Score = 50% goal completion + 50% information security (concealing goals)

@kbench.task(name='barterbench')
def barterbench(llm, seed: int) -> float:
    """Competitive N-agent marketplace. Score = 0.5*GC + 0.5*ISS. GC: target items acquired. ISS: whether you concealed your goals from competitors. Most LLMs score ISS=0% (cooperative norm failure). Beat the heuristic baseline."""
    results = [run_match(llm, s, seed=seed) for s in SCENARIOS]
    composite_gc = sum(r['gc'] for r in results) / len(results)
    avg_iss      = sum(r['iss'] for r in results) / len(results)
    score        = 0.5 * composite_gc + 0.5 * avg_iss

    print(f"Seed {seed} | Score: {score:.1%} | GC: {composite_gc:.1%} | ISS: {avg_iss:.1%}")
    for r in results:
        print(f"  {r['scenario']}: GC={r['gc']:.1%}  ISS={r['iss']:.1%}")

    return score

In [ ]:
%choose barterbench

In [ ]:
# 5 seeds × 1 task (3 scenarios internally) = 5 task runs per model
# ~135 LLM calls per model, well within $10 daily quota
SEEDS = list(range(5))

for seed in SEEDS:
    barterbench.run(llm=kbench.llm, seed=seed)